In [ ]:
# ============================================================
# GENERIC PANDAS DATA CLEANING WORKFLOW
# ============================================================
#
# WORKFLOW
# --------
#
# 1. Charger les données
#
#       df = pd.read_csv(...)
#
# 2. Inspecter le dataset
#
#       inspect_dataframe(df)
#
# 3. Détecter automatiquement les types
#
#       numeric_cols = detect_numeric_columns(df)
#       date_cols = detect_date_columns(df)
#       category_cols = detect_category_columns(df)
#
# 4. Vérifier les détections
#
#       print(numeric_cols)
#       print(date_cols)
#       print(category_cols)
#
#       Exemple :
#
#       zipcode détecté comme numeric ?
#       => le retirer manuellement
#
#       numeric_cols.remove("zipcode")
#
# 5. Convertir les types
#
#       df = convert_numeric_columns(
#           df,
#           numeric_cols
#       )
#
#       df = convert_date_columns(
#           df,
#           date_cols
#       )
#
#       df = convert_category_columns(
#           df,
#           category_cols
#       )
#
#       df = clean_remaining_text(df)
#
# 6. Vérification finale
#
#       df.info()
#       df.dtypes
#       df.isnull().sum()
#
# 7. Analyse
#
#       - EDA
#       - Visualisations
#       - Statistiques
#       - Machine Learning
#
#
# PIPELINE MENTAL
# ---------------
#
# Charger
#    ↓
# Inspecter
#    ↓
# Détecter
#    ↓
# Valider
#    ↓
# Convertir
#    ↓
# Vérifier
#    ↓
# Analyser
#
# ============================================================

import pandas as pd


# ============================================================
# INSPECTION
# ============================================================

def inspect_dataframe(df):

    print("\n" + "=" * 60)
    print("DATASET OVERVIEW")
    print("=" * 60)

    print(f"\nShape : {df.shape}")

    print("\nDtypes :")
    print(df.dtypes)

    print("\nMissing Values :")
    print(df.isnull().sum())

    print("\nMemory Usage :")
    print(
        round(
            df.memory_usage(deep=True).sum() / 1024**2,
            2
        ),
        "MB"
    )

    print("\nPreview :")
    print(df.head())


# ============================================================
# CONVERSIONS UNITAIRES
# ============================================================

def to_int(series):
    """
    Entier nullable Pandas.
    Supporte les valeurs manquantes.
    """

    return (
        pd.to_numeric(
            series,
            errors="coerce"
        )
        .astype("Int64")
    )


def to_float(series):
    """
    Conversion float.
    """

    return pd.to_numeric(
        series,
        errors="coerce"
    )


def clean_text(series):
    """
    Nettoyage texte standard.
    """

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )


def clean_id(series):
    """
    Nettoyage identifiants.
    """

    return (
        series
        .astype(str)
        .str.strip()
        .str.upper()
    )


def to_datetime_col(series):
    """
    Conversion datetime.
    """

    return pd.to_datetime(
        series,
        errors="coerce"
    )


# ============================================================
# DETECTION AUTOMATIQUE
# ============================================================

def detect_numeric_columns(
    df,
    threshold=0.80
):
    """
    Détecte les colonnes majoritairement numériques.
    """

    numeric_cols = []

    for col in df.columns:

        converted = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        success_rate = converted.notna().mean()

        if success_rate >= threshold:
            numeric_cols.append(col)

    return numeric_cols


def detect_date_columns(
    df,
    threshold=0.80
):
    """
    Détecte les colonnes majoritairement dates.
    """

    date_cols = []

    for col in df.columns:

        converted = pd.to_datetime(
            df[col],
            errors="coerce"
        )

        success_rate = converted.notna().mean()

        if success_rate >= threshold:
            date_cols.append(col)

    return date_cols


def detect_category_columns(
    df,
    max_unique_ratio=0.10
):
    """
    Détecte les colonnes candidates au type category.
    """

    category_cols = []

    object_cols = df.select_dtypes(
        include="object"
    )

    for col in object_cols:

        ratio = (
            df[col].nunique()
            /
            len(df)
        )

        if ratio <= max_unique_ratio:
            category_cols.append(col)

    return category_cols


# ============================================================
# CONVERSIONS PAR GROUPE
# ============================================================

def convert_numeric_columns(
    df,
    numeric_cols
):

    for col in numeric_cols:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    return df


def convert_date_columns(
    df,
    date_cols
):

    for col in date_cols:

        df[col] = pd.to_datetime(
            df[col],
            errors="coerce"
        )

    return df


def convert_category_columns(
    df,
    category_cols
):

    for col in category_cols:

        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.lower()
            .astype("category")
        )

    return df


def clean_remaining_text(df):
    """
    Nettoie les colonnes texte restantes.
    """

    object_cols = df.select_dtypes(
        include="object"
    )

    for col in object_cols:

        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
        )

    return df


# ============================================================
# PIPELINE COMPLET
# ============================================================

def auto_clean_dataframe(df):

    print("\nDetecting column types...\n")

    numeric_cols = detect_numeric_columns(df)

    date_cols = detect_date_columns(df)

    category_cols = detect_category_columns(df)

    print("Numeric Columns:")
    print(numeric_cols)

    print("\nDate Columns:")
    print(date_cols)

    print("\nCategory Columns:")
    print(category_cols)

    # ------------------------------------------------
    # IMPORTANT
    # Vérification humaine avant conversion
    # ------------------------------------------------
    #
    # Exemple :
    #
    # numeric_cols.remove("zipcode")
    #
    # si zipcode est un code postal
    # et non une quantité.
    #
    # ------------------------------------------------

    df = convert_numeric_columns(
        df,
        numeric_cols
    )

    df = convert_date_columns(
        df,
        date_cols
    )

    df = convert_category_columns(
        df,
        category_cols
    )

    df = clean_remaining_text(df)

    return df


# ============================================================
# UTILISATION
# ============================================================

df = pd.read_csv("data.csv")

# Inspection

inspect_dataframe(df)

# Nettoyage automatique

df = auto_clean_dataframe(df)

# Vérification finale

print("\n" + "=" * 60)
print("FINAL DTYPES")
print("=" * 60)

print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nMemory Usage:")
print(
    round(
        df.memory_usage(deep=True).sum()
        / 1024**2,
        2
    ),
    "MB"
)

print("\nPreview:")
print(df.head())


# ============================================================
# DECISION TREE
# ============================================================
#
# Pour chaque colonne :
#
# Est-ce un nombre ?
#     ↓
# Numeric
#
# Est-ce une date ?
#     ↓
# Datetime
#
# Est-ce une catégorie répétée ?
#     ↓
# Category
#
# Est-ce un identifiant ?
#     ↓
# String
#
# Est-ce du texte libre ?
#     ↓
# String
#
# ============================================================